In [ ]:
# -*- coding: utf-8 -*-
import os, time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from PIL import Image
import pandas as pd

# 설치: pip install img2table opencv-python-headless pandas

from img2table.document import Image as I2TImage
from img2table.ocr import TesseractOCR

#  설정
TICKER = "TSLA"
CATEGORY = "key-financial-ratios"
SAVE_FOLDER = "screenshots"
STITCHED_PATH = "stitched_full.png"
OUTPUT_EXCEL = f"{TICKER}_{CATEGORY}_ocr.xlsx"
URL = f"https://www.macrotrends.net/stocks/charts/{TICKER}/tesla/{CATEGORY}?freq=Q"

# 1. Selenium: 페이지 열고 스크롤하며 스크린샷 저장
os.makedirs(SAVE_FOLDER, exist_ok=True)

opts = Options()
opts.add_argument("--window-size=1600,1200")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)
driver.get(URL)
time.sleep(3)  # 페이지 로딩

grid = driver.find_element(By.CSS_SELECTOR, "div[id^='contenttablejqxgrid']")
total_w = driver.execute_script("return arguments[0].scrollWidth", grid)
view_w = driver.execute_script("return arguments[0].clientWidth", grid)

step = int(view_w * 0.8)
num_steps = (total_w - view_w) // step + 1

image_files = []
for i in range(num_steps + 1):
    x = i * step
    driver.execute_script("arguments[0].scrollLeft = arguments[1];", grid, x)
    time.sleep(1)
    fname = os.path.join(SAVE_FOLDER, f"part_{i:02d}.png")
    driver.save_screenshot(fname)
    image_files.append(fname)
    print(f"Saved screenshot: {fname}")

driver.quit()

# 2. 이미지 이어붙이기 (가로)
images = [Image.open(f) for f in image_files]
widths, heights = zip(*(img.size for img in images))
total_w = sum(widths)
max_h = max(heights)

stitched = Image.new("RGB", (total_w, max_h))
x_offset = 0
for img in images:
    stitched.paste(img, (x_offset, 0))
    x_offset += img.width

stitched.save(STITCHED_PATH)
print("Stitched image saved:", STITCHED_PATH)

# 3. img2table OCR 파싱
ocr = TesseractOCR(n_threads=4, lang="eng")
doc = I2TImage(STITCHED_PATH)
tables = doc.extract_tables(ocr=ocr, min_confidence=50)

if not tables:
    print("No table detected by img2table.")
else:
    df = tables[0].df
    df.to_excel(OUTPUT_EXCEL, index=False)
    print("Extracted table saved to:", OUTPUT_EXCEL)
